# 10. 언어 모델 (Language Model)

이번 장에서는 자연어 처리의 가장 기초가 되는 **언어 모델(Language Model, LM)**을 다룬다. 언어 모델이 무엇인지, 왜 단어 시퀀스에 확률을 할당해야 하는지부터 시작해, 전통적 접근인 **통계적 언어 모델(SLM)**과 그 한계를 보완한 **N-gram 언어 모델**을 거쳐, 언어 모델의 성능을 수치화하는 평가 지표인 **펄플렉서티(Perplexity, PPL)**까지 정리한다.

> **이 장의 흐름** : 언어 모델의 정의 → 단어 시퀀스의 확률 할당 → 조건부 확률과 연쇄 법칙으로 다음 단어 예측 → 카운트 기반 SLM → 희소 문제 → N-gram 언어 모델과 그 한계 → 펄플렉서티(PPL) → 통계 기반 vs 인공 신경망 언어 모델.

> 이 장은 이론 중심이지만, 핵심 개념은 짧은 파이썬 코드로 직접 확인하며 넘어간다. 외부 데이터 없이 동작하는 장난감(toy) 예제로 카운트·확률·PPL이 어떻게 계산되는지 손으로 따라가 본다.

# 1. 언어 모델이란?

**언어 모델(Language Model, LM)**은 언어라는 현상을 모델링하고자 **단어 시퀀스(문장)에 확률을 할당(assign)**하는 모델이다. 풀어 쓰면, 언어 모델은 **가장 자연스러운 단어 시퀀스를 찾아내는 모델**이다.

언어 모델을 만드는 방법은 크게 **통계를 이용한 방법**과 **인공 신경망을 이용한 방법**으로 나뉜다. 최근에는 인공 신경망을 이용한 방법이 더 좋은 성능을 보이며, GPT나 BERT 같은 최신 기술 또한 인공 신경망 언어 모델의 개념 위에 만들어졌다. 이 장에서는 먼저 언어 모델의 개념과 전통적 접근인 통계적 언어 모델을 다룬다.

## 1-1. 언어 모델과 언어 모델링

단어 시퀀스에 확률을 할당하기 위해 **가장 보편적으로 사용되는 방법은 이전 단어들이 주어졌을 때 다음 단어를 예측하도록 하는 것**이다.

> 다른 유형으로는 양쪽 단어로부터 가운데 비어 있는 단어를 예측하는 모델도 있다. 이는 빈칸 추론 문제와 비슷하며 **BERT** 챕터에서 다룬다. 그때까지는 **이전 단어들로부터 다음 단어를 예측**하는 방식에 집중한다.

언어 모델에 -ing를 붙인 **언어 모델링(Language Modeling)**은 주어진 단어들로부터 아직 모르는 단어를 예측하는 작업을 말한다. 즉, 이전 단어들로부터 다음 단어를 예측하는 일이 곧 언어 모델링이다. 스탠퍼드 대학교에서는 언어 모델을 **문법(grammar)**에 비유하기도 하는데, 단어들의 조합이 얼마나 적절한지를 알려주는 일이 문법이 하는 일과 닮았기 때문이다.

## 1-2. 단어 시퀀스의 확률 할당이 필요한 이유

자연어 처리에서 단어 시퀀스에 확률을 할당하는 일이 왜 필요할까? 대문자 $P$ 를 확률이라 할 때, 다음과 같은 작업들이 확률을 활용한다.

**a. 기계 번역(Machine Translation)** — 언어 모델은 두 후보 문장을 비교하여 더 자연스러운 쪽에 높은 확률을 준다.

$$ P(\text{나는 버스를 탔다}) > P(\text{나는 버스를 태운다}) $$

**b. 오타 교정(Spell Correction)** — 문맥상 더 적절한 문장에 높은 확률을 준다.

$$ P(\text{달려갔다}) > P(\text{잘려갔다}) $$

**c. 음성 인식(Speech Recognition)** — 발음이 비슷한 후보 중 더 자연스러운 문장을 고른다.

$$ P(\text{나는 메론을 먹는다}) > P(\text{나는 메롱을 먹는다}) $$

이처럼 언어 모델은 확률을 통해 더 적절한 문장을 판단한다.

## 1-3. 다음 단어 예측: 조건부 확률과 연쇄 법칙

하나의 단어를 $w$, 단어 시퀀스를 대문자 $W$ 라 하면, $n$ 개의 단어가 등장하는 단어 시퀀스 $W$ 의 확률은 다음과 같다.

$$ P(W) = P(w_1, w_2, w_3, \ldots, w_n) $$

**다음 단어 등장 확률**은 이전 단어들이 주어졌을 때의 조건부 확률로 표현한다. $n-1$ 개의 단어가 나열된 상태에서 $n$ 번째 단어의 확률은 다음과 같다(기호 $|$ 는 조건부 확률을 의미한다).

$$ P(w_n \mid w_1, \ldots, w_{n-1}) $$

전체 단어 시퀀스 $W$ 의 확률은 모든 단어가 예측되고 나서야 알 수 있으므로, 각 단어가 이전 단어들이 주어졌을 때 등장할 확률의 **곱**으로 구성된다. 이를 조건부 확률의 **연쇄 법칙(chain rule)**이라 한다.

$$ P(W) = P(w_1, w_2, \ldots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, \ldots, w_{i-1}) $$

> **연쇄 법칙의 직관** : 두 확률에 대해 $P(A, B) = P(A)\,P(B \mid A)$ 가 성립한다. 이를 4개로 일반화하면 $P(A,B,C,D) = P(A)\,P(B \mid A)\,P(C \mid A,B)\,P(D \mid A,B,C)$ 가 된다. 문장의 확률도 같은 방식으로 각 단어를 하나씩 조건부로 풀어 곱한 것이다.

## 1-4. 언어 모델의 직관과 검색 엔진 예시

"비행기를 타려고 공항에 갔는데 지각을 하는 바람에 비행기를 [?]"라는 문장에서, 사람은 '비행기를' 다음에 쉽게 **'놓쳤다'**를 떠올린다. 여러 후보 단어 중 '놓쳤다'의 확률이 가장 높다고 판단했기 때문이다.

기계도 마찬가지다. 앞에 어떤 단어들이 나왔는지 고려하여 후보 단어들의 등장 확률을 추정하고, 가장 높은 확률을 가진 단어를 선택한다. 우리가 매일 사용하는 **검색 엔진의 자동완성**이 바로 입력된 단어 나열에 대해 다음 단어를 예측하는 언어 모델의 대표적인 예다.

# 2. 통계적 언어 모델 (SLM)과 N-gram

**통계적 언어 모델(Statistical Language Model, SLM)**은 통계적인 접근으로 언어를 모델링한다. 이 절에서는 문장의 확률을 카운트(count)에 기반해 구하는 방법과 그 한계인 **희소 문제**, 그리고 이를 완화하는 **N-gram 언어 모델**을 살펴본다.

## 2-1. 문장의 확률: 조건부 확률의 연쇄 법칙

각 단어는 문맥이라는 관계로 인해 이전 단어의 영향을 받아 등장한다. 따라서 문장의 확률은 각 단어가 이전 단어가 주어졌을 때 다음 단어로 등장할 확률의 **곱**으로 구성된다.

$$ P(w_1, w_2, \ldots, w_n) = \prod_{i=1}^{n} P(w_i \mid w_1, \ldots, w_{i-1}) $$

예를 들어 문장 'An adorable little boy is spreading smiles'의 확률은 다음과 같이 분해된다.

$$
\begin{aligned}
P(\text{An adorable little boy is spreading smiles}) =\ & P(\text{An}) \times P(\text{adorable} \mid \text{An}) \\
& \times P(\text{little} \mid \text{An adorable}) \times P(\text{boy} \mid \text{An adorable little}) \\
& \times P(\text{is} \mid \text{An adorable little boy}) \times \cdots
\end{aligned}
$$

즉, 문장의 확률을 구하려면 각 단어에 대한 예측 확률들을 모두 곱하면 된다.

## 2-2. 카운트 기반의 접근

그렇다면 SLM은 이전 단어로부터 다음 단어의 확률을 어떻게 구할까? 정답은 **카운트(count)에 기반**해 계산하는 것이다. 'An adorable little boy'가 나왔을 때 'is'가 나올 확률은 다음과 같다.

$$ P(\text{is} \mid \text{An adorable little boy}) = \frac{\text{count}(\text{An adorable little boy is})}{\text{count}(\text{An adorable little boy})} $$

예를 들어 코퍼스에서 'An adorable little boy'가 100번 등장했고 그 다음에 'is'가 등장한 경우가 30번이라면, 이 확률은 **30%**가 된다. 작은 장난감 코퍼스로 이 메커니즘을 직접 구현해 본다.

In [1]:
from collections import Counter

# 토큰 시퀀스에서 n-gram 빈도를 세는 함수
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

# P(word | context) = count(context + word) / count(context)
def count_based_prob(context, word, tokens):
    ctx = tuple(context.split())
    full_counts = ngram_counts(tokens, len(ctx) + 1)
    ctx_counts = ngram_counts(tokens, len(ctx))
    full = full_counts[ctx + (word,)]
    denom = ctx_counts[ctx]
    return full / denom if denom > 0 else 0.0

# 교안 예시 재현: 'An adorable little boy' 100번, 그 뒤 'is' 30번 -> 0.3
count_context = 100   # count("An adorable little boy")
count_full = 30       # count("An adorable little boy is")
print('P(is | An adorable little boy) =', count_full / count_context)


P(is | An adorable little boy) = 0.3


## 2-3. 희소 문제 (Sparsity Problem)

언어 모델의 목표는 많은 코퍼스를 훈련시켜 **현실의 확률 분포를 근사**하는 것이다. 그런데 카운트 기반 접근은 정말 방대한 양의 데이터를 필요로 한다.

만약 코퍼스에 'An adorable little boy is'라는 시퀀스가 한 번도 없었다면 이 시퀀스의 확률은 **0**이 되고, 'An adorable little boy'라는 시퀀스조차 없었다면 분모가 0이 되어 **확률이 정의되지 않는다**. 하지만 현실에서 이 시퀀스는 분명 존재하고 문법에도 맞으므로, 0이라고 단정하는 것은 정확한 모델링이 아니다.

> 이처럼 **충분한 데이터를 관측하지 못해 언어를 정확히 모델링하지 못하는 문제**를 **희소 문제(sparsity problem)**라고 한다. 스무딩(smoothing), 백오프(back-off) 같은 일반화 기법으로 완화할 수는 있지만 근본적인 해결책은 되지 못한다. 결국 이 한계 때문에 언어 모델의 트렌드는 통계적 언어 모델에서 인공 신경망 언어 모델로 넘어간다.

In [2]:
# 희소 문제 시연: 코퍼스에 없는 시퀀스는 카운트가 0이 된다
corpus = "an adorable little boy is spreading smiles".split()
counts4 = ngram_counts(corpus, 4)

seq_seen = ('an', 'adorable', 'little', 'boy')        # 코퍼스에 존재
seq_unseen = ('an', 'adorable', 'little', 'girl')     # 코퍼스에 없음

print('count(an adorable little boy)  =', counts4[seq_seen])    # 1
print('count(an adorable little girl) =', counts4[seq_unseen])  # 0 -> 확률 0 (희소 문제)

# 분모가 0이면 확률 자체가 정의되지 않는다
denom = ngram_counts(corpus, 3)[('an', 'adorable', 'kind')]
print('정의 가능 여부:', '정의되지 않음 (분모=0)' if denom == 0 else f'{denom}')


count(an adorable little boy)  = 1
count(an adorable little girl) = 0
정의 가능 여부: 정의되지 않음 (분모=0)


## 2-4. N-gram 언어 모델

**N-gram 언어 모델**은 여전히 카운트에 기반하므로 SLM의 일종이다. 다만 이전에 등장한 **모든 단어를 고려하는 것이 아니라 일부 단어만 고려**한다. 이때 **몇 개의 단어를 볼지 결정하는 것이 n**이다.

핵심 아이디어는 참고하는 단어 수를 줄이면 코퍼스에서 카운트할 수 있는 가능성이 높아진다는 것이다. 예컨대 'An adorable little boy is'라는 긴 시퀀스보다 'boy is'라는 짧은 시퀀스가 코퍼스에 존재할 가능성이 더 높다.

$$ P(\text{is} \mid \text{An adorable little boy}) \approx P(\text{is} \mid \text{boy}) $$

**n-gram은 n개의 연속적인 단어 나열**을 의미한다. n=1은 **유니그램(unigram)**, n=2는 **바이그램(bigram)**, n=3은 **트라이그램(trigram)**이라 하고, n이 4 이상이면 숫자를 그대로 붙여 4-gram처럼 부른다. 문장 'an adorable little boy is spreading smiles'에서 각 n-gram을 직접 추출해 본다.

In [3]:
sentence = "an adorable little boy is spreading smiles"
tokens = sentence.split()

# n개의 연속된 단어 묶음(n-gram)을 모두 추출
def make_ngrams(tokens, n):
    return [tokens[i:i+n] for i in range(len(tokens) - n + 1)]

for n, name in [(1, 'unigrams'), (2, 'bigrams'), (3, 'trigrams'), (4, '4-grams')]:
    grams = [' '.join(g) for g in make_ngrams(tokens, n)]
    print(f'{name:9s}: {grams}')


unigrams : ['an', 'adorable', 'little', 'boy', 'is', 'spreading', 'smiles']
bigrams  : ['an adorable', 'adorable little', 'little boy', 'boy is', 'is spreading', 'spreading smiles']
trigrams : ['an adorable little', 'adorable little boy', 'little boy is', 'boy is spreading', 'is spreading smiles']
4-grams  : ['an adorable little boy', 'adorable little boy is', 'little boy is spreading', 'boy is spreading smiles']


## 2-5. N-gram 모델의 동작 예시

n-gram을 통한 언어 모델에서 다음 단어 예측은 오직 **n-1개의 단어에만 의존**한다. 예를 들어 4-gram 모델로 'boy is spreading' 다음 단어를 예측한다면, 직전 3개 단어만 보고 다음 단어를 고른다.

$$ P(w \mid \text{boy is spreading}) = \frac{\text{count}(\text{boy is spreading } w)}{\text{count}(\text{boy is spreading})} $$

코퍼스에서 'boy is spreading'이 1,000번, 'boy is spreading insults'가 500번, 'boy is spreading smiles'가 200번 등장했다고 하면 다음과 같이 계산된다.

In [4]:
# 교안 예시 재현: boy is spreading 다음 단어 예측 (4-gram)
count_context = {('boy', 'is', 'spreading'): 1000}
count_full = {
    ('boy', 'is', 'spreading', 'insults'): 500,
    ('boy', 'is', 'spreading', 'smiles'): 200,
}

context = ('boy', 'is', 'spreading')
for word in ['insults', 'smiles']:
    p = count_full[context + (word,)] / count_context[context]
    print(f'P({word} | boy is spreading) = {p:.3f}')

best = max(['insults', 'smiles'],
          key=lambda w: count_full[context + (w,)] / count_context[context])
print('-> 모델의 선택:', best, '(확률이 더 높은 쪽)')


P(insults | boy is spreading) = 0.500
P(smiles | boy is spreading) = 0.200
-> 모델의 선택: insults (확률이 더 높은 쪽)


## 2-6. N-gram 언어 모델의 한계

n-gram은 **앞 단어 몇 개만 보기 때문에** 의도한 대로 문장을 끝맺지 못하는 경우가 생긴다. 위 예시에서 모델은 'insults(모욕)'를 선택하지만, '작고 사랑스러운(an adorable little)'이라는 수식어까지 고려했다면 '웃음을 지었다(smiles)'가 더 자연스러울 수 있다. 즉, **전체 문장을 고려하는 모델보다 정확도가 떨어질 수밖에 없다.**

**한계 1 — 희소 문제(Sparsity Problem)** : 일부 단어만 보아 카운트 가능성을 높였지만, n-gram에 대한 희소 문제는 여전히 존재한다.

**한계 2 — n 선택의 trade-off** : n을 크게 잡으면 해당 n-gram을 카운트할 확률이 낮아져 희소 문제가 심해지고 모델 크기도 커진다. 반대로 n을 작게 잡으면 카운트는 잘 되지만 근사의 정확도가 현실 분포와 멀어진다. 그래서 **권장되는 n은 최대 5를 넘지 않는다.**

> 스탠퍼드 공유 자료에 따르면, 월스트리트 저널의 3,800만 개 토큰으로 학습하고 1,500만 개 테스트 데이터로 평가했을 때 다음과 같은 펄플렉서티(PPL)가 나왔다. PPL은 **낮을수록 좋은 성능**을 뜻한다.

| | Unigram | Bigram | Trigram |
|:--|:--:|:--:|:--:|
| **Perplexity** | 962 | 170 | 109 |

n을 1에서 2, 2에서 3으로 올릴 때마다 성능(낮은 PPL)이 좋아지는 것을 확인할 수 있다. 결국 이러한 본질적 취약점 때문에, n-gram보다 대체로 성능이 우수한 **인공 신경망 언어 모델**이 대안으로 많이 사용된다.

# 3. 펄플렉서티 (Perplexity, PPL)

두 모델 A, B의 성능을 비교하려면 실제 작업(번역·오타 교정 등)에 투입해 정확도를 비교할 수 있지만, 이는 공수가 너무 크다. 모델 수가 늘어나면 시간도 배로 늘어난다. 이보다 다소 부정확할 수는 있어도 **테스트 데이터에 대해 빠르게 식으로 계산되는 평가 방법**이 바로 **펄플렉서티(perplexity, 줄여서 PPL)**다.

## 3-1. PPL: 언어 모델의 평가 지표

영어 'perplexed'는 '헷갈리는'과 유사한 의미를 가진다. 즉 PPL은 모델이 **헷갈리는 정도**로 이해하면 된다. 처음 배울 때 낯선 점은, **PPL은 수치가 낮을수록 언어 모델의 성능이 좋다**는 것이다.

PPL은 **문장의 길이로 정규화된 문장 확률의 역수**다. 문장의 길이를 $N$ 이라 할 때 PPL은 다음과 같다.

$$ PPL(W) = P(w_1, w_2, \ldots, w_N)^{-\frac{1}{N}} = \sqrt[N]{\frac{1}{P(w_1, w_2, \ldots, w_N)}} $$

문장의 확률에 연쇄 법칙을 적용하면 다음과 같다.

$$ PPL(W) = \sqrt[N]{\frac{1}{\prod_{i=1}^{N} P(w_i \mid w_1, \ldots, w_{i-1})}} $$

여기에 n-gram을 적용할 수도 있다. bigram 언어 모델의 경우 식은 다음과 같다.

$$ PPL(W) = \sqrt[N]{\frac{1}{\prod_{i=1}^{N} P(w_i \mid w_{i-1})}} $$

> 분모의 확률 곱이 매우 작아지면 컴퓨터에서 언더플로(underflow)가 발생할 수 있다. 이 때문에 실제 구현에서는 확률을 곱하는 대신 **로그(log)를 취해 더하는 방식**으로 계산한다. 아래 코드도 이 방식을 따른다.

In [5]:
import math

# PPL(W) = P(W)^(-1/N) = exp( -(1/N) * sum(log P(w_i | ...)) )
def perplexity(probs):
    N = len(probs)
    log_sum = sum(math.log(p) for p in probs)   # log P(W)
    return math.exp(-log_sum / N)

# 모든 시점에서 다음 단어 확률이 1/10로 동일한 경우
probs = [1/10] * 5
print('각 시점 확률 1/10, 길이 5 -> PPL =', round(perplexity(probs), 1))


각 시점 확률 1/10, 길이 5 -> PPL = 10.0


## 3-2. 분기 계수 (Branching factor)

PPL은 선택할 수 있는 가능한 경우의 수를 의미하는 **분기 계수(branching factor)**다. 즉, 언어 모델이 특정 시점에서 **평균적으로 몇 개의 선택지를 가지고 고민하는지**를 나타낸다.

예를 들어 어떤 테스트 데이터에 대해 PPL이 10이 나왔다면, 그 모델은 다음 단어를 예측하는 매 시점마다 **평균 10개의 단어**를 두고 어느 것이 정답인지 고민한다고 볼 수 있다. 모든 시점에서 확률이 균등하게 $\frac{1}{10}$ 이라고 가정하면 다음이 성립한다.

$$ PPL(W) = \left(\frac{1}{10}^{\,N}\right)^{-\frac{1}{N}} = \left(\frac{1}{10}\right)^{-1} = 10 $$

같은 테스트 데이터로 두 모델의 PPL을 비교하면, **PPL이 더 낮은 모델의 성능이 더 좋다**고 본다.

In [6]:
# 분기 계수 직관: 매 시점 후보가 V개로 균등하면 PPL은 V가 된다
for V in [5, 10, 50]:
    probs = [1/V] * 8          # 길이 8 문장, 매 시점 균등 확률 1/V
    print(f'후보 {V:2d}개로 균등 -> PPL = {perplexity(probs):.1f}')

print()
# 같은 테스트 데이터로 두 모델 비교 (확률이 높을수록 = 덜 헷갈릴수록 PPL이 낮다)
model_A = [0.5, 0.4, 0.5, 0.3]   # 더 확신 있는 예측
model_B = [0.1, 0.2, 0.1, 0.2]   # 더 헷갈리는 예측
print('Model A PPL =', round(perplexity(model_A), 2))
print('Model B PPL =', round(perplexity(model_B), 2))
print('-> 성능이 더 좋은 모델:', 'A' if perplexity(model_A) < perplexity(model_B) else 'B')


후보  5개로 균등 -> PPL = 5.0
후보 10개로 균등 -> PPL = 10.0
후보 50개로 균등 -> PPL = 50.0

Model A PPL = 2.4
Model B PPL = 7.07
-> 성능이 더 좋은 모델: A


## 3-3. 기존 언어 모델 vs 인공 신경망 언어 모델

주의할 점이 있다. **PPL이 낮다는 것은 테스트 데이터 상에서 높은 정확도를 보인다는 의미이지, 사람이 느끼기에 반드시 좋은 언어 모델이라는 뜻은 아니다.** 또한 PPL은 테스트 데이터에 의존하므로, 두 개 이상의 모델을 비교할 때는 정량적으로 양이 많고 도메인에 알맞은 **동일한 테스트 데이터**를 사용해야 신뢰도가 높다.

페이스북 AI 연구팀은 n-gram 언어 모델과 딥러닝 기반 언어 모델을 PPL로 비교한 표를 공개한 바 있다.

| Model | Perplexity |
|:--|:--:|
| Interpolated Kneser-Ney 5-gram (Chelba et al., 2013) | 67.6 |
| RNN-1024 + MaxEnt 9-gram (Chelba et al., 2013) | 51.3 |
| RNN-2048 + BlackOut sampling (Ji et al., 2015) | 68.3 |
| Sparse Non-negative Matrix factorization (Shazeer et al., 2015) | 52.9 |
| LSTM-2048 (Jozefowicz et al., 2016) | 43.7 |
| 2-layer LSTM-8192 (Jozefowicz et al., 2016) | 30 |
| Ours small (LSTM-2048) | 43.9 |
| Ours large (2-layer LSTM-2048) | 39.8 |

맨 윗줄은 n-gram 기반 모델로 PPL이 **67.6**이며, 일반화 기법인 Interpolated Kneser-Ney가 적용된 5-gram 모델이다. 그 아래는 모두 인공 신경망 기반 모델이다. 아직 RNN·LSTM이 무엇인지 배우지 않았지만, **인공 신경망을 이용한 언어 모델 대부분이 n-gram 기반 모델보다 더 좋은 성능(낮은 PPL)을 받았음**을 확인할 수 있다.

---

이로써 10장 **언어 모델**의 핵심을 정리했다. 언어 모델이 단어 시퀀스에 확률을 할당하는 일을 한다는 정의에서 출발해, 조건부 확률의 연쇄 법칙으로 다음 단어를 예측하는 원리, 카운트 기반 SLM과 그 한계인 희소 문제, 이를 완화한 N-gram 언어 모델과 trade-off, 그리고 성능을 수치화하는 평가 지표 펄플렉서티(PPL)까지 살펴봤다.

핵심은 **통계적 언어 모델은 희소 문제라는 본질적 한계를 가지며, 이를 극복하기 위한 대안으로 인공 신경망 언어 모델로 트렌드가 넘어간다**는 것이다. 다음 장부터는 이 인공 신경망을 이용한 접근을 본격적으로 다룬다.